# IT4653 - Đề tài 3 - Notebook chạy trực tiếp trên Kaggle

Toàn bộ code nằm trong notebook này để cả nhóm đọc, sửa một cell và chạy lại ngay. Trước khi chạy: chọn GPU T4 x2, Add Input dataset `pankrzysiu/cifar10-python`, rồi chọn `RUN_ALL_CONFIGS` hoặc `MEMBER`, `DEBUG`, `PART` ở cell đầu.

## 1. Import và các ô nhóm thường thay đổi

In [ ]:
import copy
import math
import platform
import random
import tarfile
import time
from datetime import datetime
from pathlib import Path

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torchvision
from torch import nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms

# ===== CHỈNH CÁC GIÁ TRỊ NÀY TRƯỚC KHI CHẠY =====
RUN_ALL_CONFIGS = False   # True: tự chạy đủ 26 cấu hình x 2 seed của cả nhóm
MEMBER = 1                 # 0: tất cả; 1: optimizer+norm; 2: schedule; 3: regularization
DEBUG = True              # True: 1 epoch/tập nhỏ; False: chạy thật 20 epoch/2 seed
PART = "pilot1"          # đổi thành part1, part2... khi chia nhiều phiên Kaggle
RUN_IDS = []              # []: chạy cả phần; hoặc ["opt_sgd"] để chạy một cấu hình
NOTEBOOK_VERSION = "v2" # v2: mỗi run có cả validation và test
SAVE_CHECKPOINTS = False  # test ngay trong run nên thường không cần lưu model
# =================================================

# Preset một công tắc cho lần chạy toàn bộ.
if RUN_ALL_CONFIGS:
    MEMBER = 0
    DEBUG = False
    PART = "all_26_configs"
    RUN_IDS = []

if MEMBER not in (0, 1, 2, 3):
    raise ValueError("MEMBER phải là 0, 1, 2 hoặc 3")
if not torch.cuda.is_available():
    raise RuntimeError("Hãy bật Accelerator > GPU trong Kaggle Settings")

GPU_NAMES = [torch.cuda.get_device_name(index) for index in range(torch.cuda.device_count())]
if any("P100" in name for name in GPU_NAMES):
    raise RuntimeError("Hãy chọn GPU T4 x2; PyTorch mặc định hiện không hỗ trợ P100 ổn định")
DEVICE = torch.device("cuda:0")  # cố ý dùng một GPU để BatchNorm luôn có cùng ý nghĩa
EPOCHS = 1 if DEBUG else 20
SEEDS = [42] if DEBUG else [42, 2026]
TRAIN_LIMIT = 512 if DEBUG else None
VAL_LIMIT = 128 if DEBUG else None
OUTPUT_DIR = Path("/kaggle/working/it4653")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Ưu tiên dataset đã giải nén; nếu Add Input là tar.gz thì giải nén vào working.
cifar_folders = list(Path("/kaggle/input").rglob("cifar-10-batches-py"))
if len(cifar_folders) == 1:
    DATA_ROOT = cifar_folders[0].parent
elif len(cifar_folders) == 0:
    archives = list(Path("/kaggle/input").rglob("cifar-10-python.tar.gz"))
    if len(archives) != 1:
        raise RuntimeError(f"Cần đúng 1 file CIFAR tar.gz, hiện tìm thấy {len(archives)}")
    DATA_ROOT = Path("/kaggle/working/cifar_data")
    if not (DATA_ROOT / "cifar-10-batches-py").exists():
        DATA_ROOT.mkdir(parents=True, exist_ok=True)
        with tarfile.open(archives[0], "r:gz") as archive:
            archive.extractall(DATA_ROOT)
else:
    raise RuntimeError(f"Chỉ Add Input một bản CIFAR-10, hiện thấy {len(cifar_folders)} bản")

if not (DATA_ROOT / "cifar-10-batches-py/data_batch_1").exists():
    raise RuntimeError("CIFAR-10 không đúng cấu trúc Python batches")

print("Python:", platform.python_version())
print("PyTorch:", torch.__version__, "| torchvision:", torchvision.__version__)
print("NumPy:", np.__version__, "| pandas:", pd.__version__, "| matplotlib:", matplotlib.__version__)
print("GPU Kaggle:", GPU_NAMES, "| notebook dùng:", DEVICE)
print("CIFAR-10 root:", DATA_ROOT)
print("Chế độ:", "DEBUG" if DEBUG else "OFFICIAL", "| epochs:", EPOCHS, "| seeds:", SEEDS)
print("Phạm vi:", "ALL 26 CONFIGS" if RUN_ALL_CONFIGS else f"MEMBER {MEMBER}")

## 2. Cấu hình gốc và danh sách thí nghiệm

Mỗi cấu hình là một dictionary phẳng. Cấu hình con chỉ ghi các giá trị khác baseline, nên nhìn vào là biết yếu tố nào đang thay đổi. Trước official `v2`, cả nhóm phải khóa danh sách này và LR; sau khi đã xem test thì không sửa cấu hình.

In [ ]:
BASE_CONFIG = {
    "optimizer": "sgd_momentum",
    "lr": 0.1,
    "momentum": 0.9,
    "weight_decay": 5e-4,
    "schedule": "constant",
    "warmup_epochs": 0,
    "step_size": 7,
    "gamma": 0.1,
    "min_lr": 1e-5,
    "normalization": "batch",
    "batch_size": 128,
    "dropout": 0.0,
    "augmentation": False,
    "early_stopping": False,
    "patience": 4,
}

def experiment(run_id, label, branches, **changes):
    """Tạo một cấu hình ngắn: id, nhãn, nhánh và những gì đổi so với baseline."""
    if isinstance(branches, str):
        branches = [branches]
    return {"id": run_id, "label": label, "branches": branches, **changes}

# Anchor được chạy một lần nhưng xuất hiện trong cả bốn phép so sánh.
ANCHOR = experiment(
    "anchor",
    "SGD momentum + WD + constant + BN/b128",
    ["optimizer", "schedule", "normalization", "regularization"],
)

MEMBER_1_EXPERIMENTS = [
    ANCHOR,
    experiment("opt_sgd", "SGD", "optimizer", optimizer="sgd", lr=0.1, momentum=0.0),
    experiment("opt_nesterov", "Nesterov", "optimizer", optimizer="nesterov", lr=0.1),
    experiment("opt_rmsprop", "RMSProp", "optimizer", optimizer="rmsprop", lr=0.001, momentum=0.0),
    experiment("opt_adam", "Adam", "optimizer", optimizer="adam", lr=0.001, momentum=None),
    experiment("opt_adamw", "AdamW", "optimizer", optimizer="adamw", lr=0.001, momentum=None),
    experiment("norm_bn_b8", "BatchNorm, batch 8", "normalization", batch_size=8),
    experiment("norm_bn_b32", "BatchNorm, batch 32", "normalization", batch_size=32),
    experiment("norm_ln_b8", "LayerNorm, batch 8", "normalization", normalization="layer", batch_size=8),
    experiment("norm_ln_b32", "LayerNorm, batch 32", "normalization", normalization="layer", batch_size=32),
    experiment("norm_ln_b128", "LayerNorm, batch 128", "normalization", normalization="layer", batch_size=128),
    experiment("norm_gn_b8", "GroupNorm, batch 8", "normalization", normalization="group", batch_size=8),
    experiment("norm_gn_b32", "GroupNorm, batch 32", "normalization", normalization="group", batch_size=32),
    experiment("norm_gn_b128", "GroupNorm, batch 128", "normalization", normalization="group", batch_size=128),
]

MEMBER_2_EXPERIMENTS = [
    experiment("sched_constant_warm", "Constant + warm-up", "schedule", warmup_epochs=2),
    experiment("sched_step", "Step decay", "schedule", schedule="step"),
    experiment("sched_step_warm", "Step decay + warm-up", "schedule", schedule="step", warmup_epochs=2),
    experiment("sched_cosine", "Cosine", "schedule", schedule="cosine"),
    experiment("sched_cosine_warm", "Cosine + warm-up", "schedule", schedule="cosine", warmup_epochs=2),
]

MEMBER_3_EXPERIMENTS = [
    experiment("reg_none", "Không dùng regularizer khảo sát", "regularization", weight_decay=0.0),
    experiment("reg_dropout_01", "Dropout 0.1", "regularization", weight_decay=0.0, dropout=0.1),
    experiment("reg_dropout_03", "Dropout 0.3", "regularization", weight_decay=0.0, dropout=0.3),
    experiment("reg_dropout_05", "Dropout 0.5", "regularization", weight_decay=0.0, dropout=0.5),
    experiment("reg_augmentation", "Data augmentation", "regularization", weight_decay=0.0, augmentation=True),
    experiment("reg_early_stop", "Early stopping", "regularization", weight_decay=0.0, early_stopping=True),
    experiment(
        "reg_combined",
        "WD + Dropout 0.3 + augmentation + early stopping",
        "regularization",
        dropout=0.3,
        augmentation=True,
        early_stopping=True,
    ),
]

EXPERIMENTS_BY_MEMBER = {
    0: MEMBER_1_EXPERIMENTS + MEMBER_2_EXPERIMENTS + MEMBER_3_EXPERIMENTS,
    1: MEMBER_1_EXPERIMENTS,
    2: MEMBER_2_EXPERIMENTS,
    3: MEMBER_3_EXPERIMENTS,
}
print("Số cấu hình riêng của thành viên 1/2/3: 14 / 5 / 7")
print("Tổng: 26 cấu hình x 2 seed = 52 lượt chạy")

## 3. Dữ liệu CIFAR-10 và split cố định

Train có thể dùng augmentation; validation và test luôn dùng transform sạch. Validation chọn checkpoint; test chỉ được đo một lần sau khi run kết thúc.

In [ ]:
CIFAR_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR_STD = (0.2470, 0.2435, 0.2616)

clean_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
])
augment_transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1),
    transforms.ToTensor(),
    transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
])

# Ba object train/validation và một object test; validation/test luôn dùng ảnh sạch.
TRAIN_CLEAN = datasets.CIFAR10(DATA_ROOT, train=True, transform=clean_transform, download=False)
TRAIN_AUGMENTED = datasets.CIFAR10(DATA_ROOT, train=True, transform=augment_transform, download=False)
EVAL_DATA = datasets.CIFAR10(DATA_ROOT, train=True, transform=clean_transform, download=False)
TEST_DATA = datasets.CIFAR10(DATA_ROOT, train=False, transform=clean_transform, download=False)

# Split seed tách khỏi training seed: mọi thí nghiệm dùng đúng cùng 45k/5k ảnh.
split_generator = torch.Generator().manual_seed(4653)
permutation = torch.randperm(len(EVAL_DATA), generator=split_generator).tolist()
OFFICIAL_TRAIN_INDICES = permutation[:45000]
OFFICIAL_VAL_INDICES = permutation[45000:]

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def make_loaders(config, seed):
    train_indices = OFFICIAL_TRAIN_INDICES
    val_indices = OFFICIAL_VAL_INDICES
    if TRAIN_LIMIT is not None:
        train_indices = train_indices[:TRAIN_LIMIT]
        val_indices = val_indices[:VAL_LIMIT]

    train_source = TRAIN_AUGMENTED if config["augmentation"] else TRAIN_CLEAN
    train_subset = Subset(train_source, train_indices)
    val_subset = Subset(EVAL_DATA, val_indices)
    generator = torch.Generator().manual_seed(seed)

    train_loader = DataLoader(
        train_subset,
        batch_size=config["batch_size"],
        shuffle=True,
        num_workers=2,
        pin_memory=True,
        generator=generator,
    )
    val_loader = DataLoader(
        val_subset,
        batch_size=256,  # batch size khảo sát chỉ áp dụng cho train
        shuffle=False,
        num_workers=2,
        pin_memory=True,
    )
    return train_loader, val_loader

def make_test_loader():
    """Test chính thức: transform sạch, không shuffle, batch eval cố định."""
    return DataLoader(
        TEST_DATA,
        batch_size=256,  # eval mode không phụ thuộc train batch; cố định để test công bằng và nhanh
        shuffle=False,
        num_workers=2,
        pin_memory=True,
    )

print("Official split:", len(OFFICIAL_TRAIN_INDICES), "train /", len(OFFICIAL_VAL_INDICES), "validation /", len(TEST_DATA), "test")

## 4. ResNet-18 cho ảnh 32×32

Stem dùng convolution 3×3, stride 1 và không max-pool. Dropout luôn đặt sau global average pooling.

In [ ]:
class LayerNorm2d(nn.Module):
    """LayerNorm trên C tại từng vị trí H×W của feature map NCHW."""

    def __init__(self, channels):
        super().__init__()
        self.norm = nn.LayerNorm(channels)

    def forward(self, inputs):
        # nn.LayerNorm chuẩn hóa chiều cuối, nên đổi NCHW → NHWC rồi đổi lại.
        outputs = inputs.permute(0, 2, 3, 1)
        outputs = self.norm(outputs)
        return outputs.permute(0, 3, 1, 2)

def make_norm(name, channels):
    if name == "batch":
        return nn.BatchNorm2d(channels)
    if name == "layer":
        return LayerNorm2d(channels)
    if name == "group":
        return nn.GroupNorm(8, channels)
    raise ValueError(f"Normalization không hợp lệ: {name}")

class BasicBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride, normalization):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, 3, stride, 1, bias=False)
        self.norm1 = make_norm(normalization, out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, 3, 1, 1, bias=False)
        self.norm2 = make_norm(normalization, out_channels)
        self.relu = nn.ReLU(inplace=True)

        # Khi shape thay đổi, shortcut 1×1 đưa x về cùng shape với nhánh chính.
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, 1, stride, bias=False),
                make_norm(normalization, out_channels),
            )
        else:
            self.shortcut = nn.Identity()

    def forward(self, inputs):
        residual = self.shortcut(inputs)
        outputs = self.relu(self.norm1(self.conv1(inputs)))
        outputs = self.norm2(self.conv2(outputs))
        return self.relu(outputs + residual)

class CifarResNet18(nn.Module):
    def __init__(self, normalization="batch", dropout=0.0, num_classes=10):
        super().__init__()
        self.current_channels = 64
        self.stem = nn.Sequential(
            nn.Conv2d(3, 64, 3, 1, 1, bias=False),
            make_norm(normalization, 64),
            nn.ReLU(inplace=True),
        )
        self.stage1 = self._make_stage(64, 2, 1, normalization)
        self.stage2 = self._make_stage(128, 2, 2, normalization)
        self.stage3 = self._make_stage(256, 2, 2, normalization)
        self.stage4 = self._make_stage(512, 2, 2, normalization)
        self.pool = nn.AdaptiveAvgPool2d((1, 1))
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(512, num_classes)
        self._initialize_weights()

    def _make_stage(self, out_channels, block_count, stride, normalization):
        blocks = [BasicBlock(self.current_channels, out_channels, stride, normalization)]
        self.current_channels = out_channels
        for _ in range(1, block_count):
            blocks.append(BasicBlock(out_channels, out_channels, 1, normalization))
        return nn.Sequential(*blocks)

    def _initialize_weights(self):
        for module in self.modules():
            if isinstance(module, nn.Conv2d):
                nn.init.kaiming_normal_(module.weight, mode="fan_out", nonlinearity="relu")
            elif isinstance(module, (nn.BatchNorm2d, nn.GroupNorm, nn.LayerNorm)):
                nn.init.ones_(module.weight)
                nn.init.zeros_(module.bias)

    def forward(self, inputs):
        outputs = self.stem(inputs)       # 32×32
        outputs = self.stage1(outputs)    # 32×32
        outputs = self.stage2(outputs)    # 16×16
        outputs = self.stage3(outputs)    # 8×8
        outputs = self.stage4(outputs)    # 4×4
        outputs = self.pool(outputs).flatten(1)
        return self.classifier(self.dropout(outputs))

def build_model(config):
    return CifarResNet18(
        normalization=config["normalization"],
        dropout=config["dropout"],
    )

## 5. Optimizer và learning-rate schedule

Scheduler được viết thành công thức theo epoch để dễ giải thích. LR được đặt ở đầu mỗi epoch.

In [ ]:
def build_optimizer(model, config):
    lr = config["lr"]
    wd = config["weight_decay"]
    momentum = config["momentum"]
    name = config["optimizer"]

    # Chỉ decay ma trận weight của Conv/Linear (ndim > 1); không decay bias/norm.
    decay_parameters = [parameter for parameter in model.parameters() if parameter.ndim > 1]
    no_decay_parameters = [parameter for parameter in model.parameters() if parameter.ndim == 1]
    parameters = [
        {"params": decay_parameters, "weight_decay": wd},
        {"params": no_decay_parameters, "weight_decay": 0.0},
    ]

    if name == "sgd":
        return torch.optim.SGD(parameters, lr=lr)
    if name == "sgd_momentum":
        return torch.optim.SGD(parameters, lr=lr, momentum=momentum)
    if name == "nesterov":
        return torch.optim.SGD(
            parameters, lr=lr, momentum=momentum, nesterov=True
        )
    if name == "rmsprop":
        return torch.optim.RMSprop(
            parameters, lr=lr, alpha=0.99, momentum=momentum
        )
    if name == "adam":
        return torch.optim.Adam(parameters, lr=lr)
    if name == "adamw":
        return torch.optim.AdamW(parameters, lr=lr)
    raise ValueError(f"Optimizer không hợp lệ: {name}")

def learning_rate_for_epoch(epoch, total_epochs, config):
    """epoch bắt đầu từ 0; warm-up chạy trước schedule chính."""
    base_lr = config["lr"]
    warmup = config["warmup_epochs"]
    if epoch < warmup:
        return base_lr * (epoch + 1) / warmup

    post_epoch = epoch - warmup
    if config["schedule"] == "constant":
        return base_lr
    if config["schedule"] == "step":
        decay_count = post_epoch // config["step_size"]
        return base_lr * (config["gamma"] ** decay_count)
    if config["schedule"] == "cosine":
        remaining = max(1, total_epochs - warmup - 1)
        progress = min(1.0, post_epoch / remaining)
        cosine = (1 + math.cos(math.pi * progress)) / 2
        return config["min_lr"] + (base_lr - config["min_lr"]) * cosine
    raise ValueError(f"Schedule không hợp lệ: {config['schedule']}")

def set_learning_rate(optimizer, learning_rate):
    for group in optimizer.param_groups:
        group["lr"] = learning_rate

## 6. Train, validation và test một lần cho mỗi lượt thí nghiệm

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, global_step):
    model.train()
    total_loss = 0.0
    total_correct = 0
    total_samples = 0
    step_rows = []

    for images, labels in loader:
        images = images.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)

        optimizer.zero_grad()
        logits = model(images)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        batch_size = labels.size(0)
        total_loss += loss.item() * batch_size
        total_correct += (logits.argmax(1) == labels).sum().item()
        total_samples += batch_size

        # Log thưa mỗi 20 step là đủ để vẽ đường hội tụ mà CSV không quá lớn.
        if global_step % 20 == 0:
            step_rows.append({
                "global_step": global_step,
                "loss": loss.item(),
                "learning_rate": optimizer.param_groups[0]["lr"],
            })
        global_step += 1

    return (
        total_loss / total_samples,
        total_correct / total_samples,
        global_step,
        step_rows,
    )

@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    total_loss = 0.0
    total_correct = 0
    total_samples = 0

    for images, labels in loader:
        images = images.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)
        logits = model(images)
        loss = criterion(logits, labels)
        batch_size = labels.size(0)
        total_loss += loss.item() * batch_size
        total_correct += (logits.argmax(1) == labels).sum().item()
        total_samples += batch_size

    return total_loss / total_samples, total_correct / total_samples

def run_experiment(config, seed):
    set_seed(seed)
    train_loader, val_loader = make_loaders(config, seed)
    model = build_model(config).to(DEVICE)
    criterion = nn.CrossEntropyLoss()
    optimizer = build_optimizer(model, config)

    history = []
    all_step_rows = []
    global_step = 0
    best_val_accuracy = -1.0
    best_state = None
    best_epoch = 0
    best_val_loss = float("inf")
    epochs_without_improvement = 0
    start_time = time.time()

    for epoch in range(config["epochs"]):
        lr = learning_rate_for_epoch(epoch, config["epochs"], config)
        set_learning_rate(optimizer, lr)
        epoch_start = time.time()

        train_loss, train_accuracy, global_step, step_rows = train_one_epoch(
            model, train_loader, criterion, optimizer, global_step
        )
        val_loss, val_accuracy = evaluate(model, val_loader, criterion)
        for row in step_rows:
            row["epoch"] = epoch + 1
        all_step_rows.extend(step_rows)

        history.append({
            "epoch": epoch + 1,
            "train_loss": train_loss,
            "train_accuracy": train_accuracy,
            "val_loss": val_loss,
            "val_accuracy": val_accuracy,
            "learning_rate": lr,
            "epoch_seconds": time.time() - epoch_start,
        })

        if val_accuracy > best_val_accuracy:
            best_val_accuracy = val_accuracy
            best_epoch = epoch + 1
            best_state = copy.deepcopy(model.state_dict())

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1

        print(
            f"{config['id']} | seed {seed} | epoch {epoch + 1:02d}/{config['epochs']} | "
            f"train loss {train_loss:.4f} | val acc {val_accuracy:.4f} | lr {lr:.6f}"
        )

        if config["early_stopping"] and epochs_without_improvement >= config["patience"]:
            print("Early stopping tại epoch", epoch + 1)
            break

    final = history[-1]
    training_seconds = time.time() - start_time

    # Pilot không được đọc test vì nhóm còn dùng pilot/validation để khóa LR và config.
    test_loss = float("nan")
    test_accuracy = float("nan")
    test_seconds = 0.0
    test_samples = 0
    if not DEBUG:
        # Cùng một quy tắc cho mọi cấu hình: validation chọn epoch đại diện.
        # Test không chọn epoch, không backward và không cập nhật model.
        model.load_state_dict(best_state)
        test_loader = make_test_loader()
        test_start = time.time()
        test_loss, test_accuracy = evaluate(model, test_loader, criterion)
        test_seconds = time.time() - test_start
        test_samples = len(test_loader.dataset)
        print(f"{config['id']} | seed {seed} | đã đo TEST một lần tại best epoch {best_epoch}")
    else:
        print("DEBUG: bỏ qua test để tránh dùng test khi còn chốt cấu hình")
    summary = {
        "experiment_id": config["id"],
        "label": config["label"],
        "branches": ",".join(config["branches"]),
        "seed": seed,
        "optimizer": config["optimizer"],
        "lr": config["lr"],
        "momentum": config["momentum"],
        "weight_decay": config["weight_decay"],
        "schedule": config["schedule"],
        "warmup_epochs": config["warmup_epochs"],
        "step_size": config["step_size"],
        "gamma": config["gamma"],
        "min_lr": config["min_lr"],
        "normalization": config["normalization"],
        "batch_size": config["batch_size"],
        "dropout": config["dropout"],
        "augmentation": config["augmentation"],
        "early_stopping": config["early_stopping"],
        "patience": config["patience"],
        "epochs_budget": config["epochs"],
        "train_samples": len(train_loader.dataset),
        "val_samples": len(val_loader.dataset),
        "test_samples": test_samples,
        "best_val_accuracy": best_val_accuracy,
        "best_epoch": best_epoch,
        "test_loss": test_loss,
        "test_accuracy": test_accuracy,
        "test_checkpoint_epoch": best_epoch,
        "checkpoint_rule": "max_val_accuracy",
        "final_train_accuracy": final["train_accuracy"],
        "final_val_accuracy": final["val_accuracy"],
        "epochs_completed": len(history),
        "training_seconds": training_seconds,
        "test_seconds": test_seconds,
        "total_seconds": training_seconds + test_seconds,
        "gpu": torch.cuda.get_device_name(0),
        "python_version": platform.python_version(),
        "torch_version": torch.__version__,
        "torchvision_version": torchvision.__version__,
        "numpy_version": np.__version__,
        "pandas_version": pd.__version__,
        "matplotlib_version": matplotlib.__version__,
        "split_seed": 4653,
        "debug": DEBUG,
        "notebook_version": NOTEBOOK_VERSION,
        "finished_at": datetime.now().isoformat(timespec="seconds"),
    }
    return summary, pd.DataFrame(history), pd.DataFrame(all_step_rows), best_state

## 7. Nhóm tự kiểm tra trực tiếp trên Kaggle

Cell này lấy một batch, chạy forward và backward thật trên GPU. Hãy tự nhìn shape và loss trước khi chạy pilot 1 epoch. Đây là kiểm tra trực tiếp, không có framework test riêng.

In [ ]:
quick_config = {**BASE_CONFIG, "id": "quick", "label": "quick", "branches": ["quick"], "epochs": 1}
quick_loader, _ = make_loaders(quick_config, seed=42)
images, labels = next(iter(quick_loader))
quick_model = build_model(quick_config).to(DEVICE)
quick_logits = quick_model(images.to(DEVICE))
quick_loss = nn.CrossEntropyLoss()(quick_logits, labels.to(DEVICE))
quick_loss.backward()
print("Batch shape:", tuple(images.shape))
print("Logits shape:", tuple(quick_logits.shape), "(phải là [batch, 10])")
print("Loss một batch:", float(quick_loss.item()))
del quick_model, quick_logits, quick_loss
torch.cuda.empty_cache()

## 8. Chạy phần của thành viên và lưu CSV

Khi `DEBUG=True` và `RUN_IDS=[]`, notebook chỉ chạy cấu hình đầu tiên với seed 42. Khi chạy thật, đổi `DEBUG=False`, đặt `PART`, và có thể chia danh sách bằng `RUN_IDS`. Nếu đặt `RUN_ALL_CONFIGS=True`, notebook tự chạy đủ 26 cấu hình × 2 seed và đo test cho từng run.

In [ ]:
selected_experiments = EXPERIMENTS_BY_MEMBER[MEMBER]
if RUN_IDS:
    selected_experiments = [item for item in selected_experiments if item["id"] in RUN_IDS]
elif DEBUG:
    selected_experiments = selected_experiments[:1]

if RUN_ALL_CONFIGS:
    selected_ids = [item["id"] for item in selected_experiments]
    if len(selected_ids) != 26 or len(set(selected_ids)) != 26:
        raise RuntimeError("Chế độ ALL phải có đúng 26 experiment ID duy nhất")
    if EPOCHS != 20 or SEEDS != [42, 2026]:
        raise RuntimeError("Chế độ ALL phải chạy 20 epoch với seed 42 và 2026")
    print("Preflight OK: 26 cấu hình x 2 seed = 52 run, mỗi run test đúng một lần")

summary_rows = []
epoch_rows = []
step_rows = []
summary_path = OUTPUT_DIR / f"summary_member{MEMBER}_{PART}.csv"
epoch_path = OUTPUT_DIR / f"epoch_log_member{MEMBER}_{PART}.csv"
step_path = OUTPUT_DIR / f"step_log_member{MEMBER}_{PART}.csv"
checkpoint_dir = OUTPUT_DIR / "checkpoints"
checkpoint_dir.mkdir(exist_ok=True)

for experiment_config in selected_experiments:
    config = {**BASE_CONFIG, **experiment_config, "epochs": EPOCHS}
    for seed in SEEDS:
        summary, history, steps, best_state = run_experiment(config, seed)
        summary_rows.append(summary)

        history["experiment_id"] = config["id"]
        history["label"] = config["label"]
        history["branches"] = ",".join(config["branches"])
        history["seed"] = seed
        history["notebook_version"] = NOTEBOOK_VERSION
        history["debug"] = DEBUG
        epoch_rows.extend(history.to_dict("records"))

        if not steps.empty:
            steps["experiment_id"] = config["id"]
            steps["label"] = config["label"]
            steps["branches"] = ",".join(config["branches"])
            steps["seed"] = seed
            steps["notebook_version"] = NOTEBOOK_VERSION
            steps["debug"] = DEBUG
            step_rows.extend(steps.to_dict("records"))

        if SAVE_CHECKPOINTS:
            torch.save(best_state, checkpoint_dir / f"{config['id']}_seed{seed}.pt")

        # Ghi lại sau từng run để run trước không mất nếu Kaggle ngắt giữa chừng.
        pd.DataFrame(summary_rows).to_csv(summary_path, index=False)
        pd.DataFrame(epoch_rows).to_csv(epoch_path, index=False)
        pd.DataFrame(step_rows).to_csv(step_path, index=False)

        # Chỉ giữ model của run hiện tại; giải phóng trước run tiếp theo.
        del best_state
        torch.cuda.empty_cache()

print("Đã lưu:")
print(summary_path)
print(epoch_path)
print(step_path)

## 9. Ghép CSV và vẽ tối thiểu 6 biểu đồ

Người tổng hợp Add Input output của ba notebook rồi chạy cell này. Cell cũng đọc CSV vừa tạo trong `/kaggle/working`.

In [ ]:
def read_matching_csv(filename_pattern):
    paths = list(Path("/kaggle/input").rglob(filename_pattern))
    paths += list(OUTPUT_DIR.rglob(filename_pattern))  # file vừa chạy được ưu tiên khi trùng key
    if not paths:
        return pd.DataFrame()
    return pd.concat([pd.read_csv(path) for path in paths], ignore_index=True)

summaries = read_matching_csv("summary_member*.csv")
epochs = read_matching_csv("epoch_log_member*.csv")
steps = read_matching_csv("step_log_member*.csv")

def keep_current_runs(frame):
    """Không trộn pilot/official hoặc log từ notebook version khác."""
    if frame.empty:
        return frame
    if "debug" not in frame or "notebook_version" not in frame:
        raise RuntimeError("CSV cũ thiếu cột debug/notebook_version; không trộn với log mới")
    is_debug = frame["debug"].astype(str).str.lower().eq("true")
    return frame[(is_debug == DEBUG) & (frame["notebook_version"] == NOTEBOOK_VERSION)]

summaries = keep_current_runs(summaries)
epochs = keep_current_runs(epochs)
steps = keep_current_runs(steps)

if summaries.empty:
    raise RuntimeError("Không có summary CSV khớp DEBUG và NOTEBOOK_VERSION hiện tại")

def require_unique_rows(frame, keys, log_name):
    """Không tự chọn giữa hai run trùng; bắt người tổng hợp bỏ đúng file thừa."""
    if frame.empty:
        return
    duplicates = frame.duplicated(keys, keep=False)
    if duplicates.any():
        examples = frame.loc[duplicates, keys].drop_duplicates().head(5).to_dict("records")
        raise RuntimeError(f"{log_name} có run trùng {examples}; hãy bỏ file cũ, không chọn ngầm")

require_unique_rows(summaries, ["experiment_id", "seed"], "summary")
require_unique_rows(epochs, ["experiment_id", "seed", "epoch"], "epoch log")
require_unique_rows(steps, ["experiment_id", "seed", "global_step"], "step log")

mean_std = (
    summaries.groupby(["experiment_id", "label"], as_index=False)
    .agg(
        best_val_accuracy_mean=("best_val_accuracy", "mean"),
        best_val_accuracy_std=("best_val_accuracy", "std"),
        test_accuracy_mean=("test_accuracy", "mean"),
        test_accuracy_std=("test_accuracy", "std"),
        seeds=("seed", "nunique"),
    )
)
mean_std.to_csv(OUTPUT_DIR / "mean_std.csv", index=False)
# Không xếp hạng chung 26 cấu hình vì mỗi nhánh trả lời một câu hỏi khác nhau.
display(mean_std.sort_values("experiment_id"))

figure_dir = OUTPUT_DIR / "figures"
figure_dir.mkdir(exist_ok=True)
accuracy_metric = "best_val_accuracy" if DEBUG else "test_accuracy"
accuracy_label = "Best validation accuracy" if DEBUG else "Test accuracy"

def has_branch(frame, branch):
    return frame["branches"].fillna("").str.contains(branch)

def finish_figure(filename, title):
    plt.title(title)
    plt.grid(alpha=0.25)
    plt.tight_layout()
    plt.savefig(figure_dir / filename, dpi=160, bbox_inches="tight")
    plt.show()

def accuracy_figure(branch, filename, title):
    data = summaries[has_branch(summaries, branch)]
    if data.empty:
        return
    stats = data.groupby(["experiment_id", "label"])[accuracy_metric].agg(["mean", "std"])
    stats = stats.sort_values("mean")
    plt.figure(figsize=(9, max(4, len(stats) * 0.45)))
    plt.barh(stats.index.get_level_values("label"), stats["mean"], xerr=stats["std"].fillna(0))
    plt.xlabel(f"{accuracy_label} (mean ± std)")
    finish_figure(filename, title)

# 1. Train và validation loss của optimizer theo epoch.
optimizer_epochs = epochs[has_branch(epochs, "optimizer")] if not epochs.empty else pd.DataFrame()
if not optimizer_epochs.empty:
    figure, axes = plt.subplots(1, 2, figsize=(13, 5), sharex=True)
    for label, group in optimizer_epochs.groupby("label"):
        curve = group.groupby("epoch")[["train_loss", "val_loss"]].mean()
        axes[0].plot(curve.index, curve["train_loss"], label=label)
        axes[1].plot(curve.index, curve["val_loss"], label=label)
    for axis, title in zip(axes, ["Train loss", "Validation loss"]):
        axis.set_title(title)
        axis.set_xlabel("Epoch")
        axis.grid(alpha=0.25)
    axes[0].set_ylabel("Loss")
    axes[1].legend(fontsize=8)
    figure.suptitle("Optimizer convergence")
    figure.tight_layout()
    figure.savefig(figure_dir / "01_optimizer_loss.png", dpi=160, bbox_inches="tight")
    plt.show()

# 2. Accuracy optimizer.
accuracy_figure("optimizer", "02_optimizer_accuracy.png", "Optimizer accuracy")

# 3. Loss schedule theo global optimizer step.
schedule_steps = steps[has_branch(steps, "schedule")] if not steps.empty else pd.DataFrame()
if not schedule_steps.empty:
    plt.figure(figsize=(10, 6))
    for label, group in schedule_steps.groupby("label"):
        curve = group.groupby("global_step")["loss"].mean().rolling(10, min_periods=1).mean()
        plt.plot(curve.index, curve.values, label=label)
    plt.xlabel("Global step")
    plt.ylabel("Train loss (rolling mean 10 points)")
    plt.legend(fontsize=8)
    finish_figure("03_schedule_loss.png", "Schedule convergence")

# 4. Accuracy schedule.
accuracy_figure("schedule", "04_schedule_accuracy.png", "Schedule accuracy")

# 5. Normalization theo batch size.
normalization_data = summaries[has_branch(summaries, "normalization")]
if not normalization_data.empty:
    plt.figure(figsize=(8, 5))
    for name, group in normalization_data.groupby("normalization"):
        stats = group.groupby("batch_size")[accuracy_metric].agg(["mean", "std"]).sort_index()
        plt.errorbar(stats.index, stats["mean"], yerr=stats["std"].fillna(0), marker="o", label=name)
    plt.xlabel("Batch size")
    plt.ylabel(accuracy_label)
    plt.legend()
    finish_figure("05_normalization.png", "Normalization × batch size")

# 6. Accuracy regularization.
accuracy_figure("regularization", "06_regularization.png", "Regularization accuracy")
print("Figures:", figure_dir)

## 10. Kiểm tra độ đầy đủ của test-all

Test đã được đo đúng một lần bên trong `run_experiment`, sau khi validation chọn checkpoint. Cell này chỉ kiểm tra đủ 26 cấu hình × 2 seed; nó không train lại và không chọn một mô hình thắng.

In [ ]:
all_experiments = MEMBER_1_EXPERIMENTS + MEMBER_2_EXPERIMENTS + MEMBER_3_EXPERIMENTS
expected_ids = {item["id"] for item in all_experiments}
seed_sets = summaries.groupby("experiment_id")["seed"].agg(lambda values: sorted(set(values)))
missing_ids = sorted(expected_ids - set(seed_sets.index))
unexpected_ids = sorted(set(seed_sets.index) - expected_ids)
wrong_seed_sets = {run_id: seeds for run_id, seeds in seed_sets.items() if seeds != [42, 2026]}
missing_test_rows = summaries["test_accuracy"].isna().sum()
invalid_test_rows = (~np.isfinite(summaries["test_loss"]) | ~np.isfinite(summaries["test_accuracy"])).sum()
wrong_test_size = (summaries["test_samples"] != 10000).sum()
wrong_checkpoint = (
    (summaries["test_checkpoint_epoch"] != summaries["best_epoch"])
    | (summaries["checkpoint_rule"] != "max_val_accuracy")
).sum()
wrong_budget = (summaries["epochs_budget"] != 20).sum()
early_stopping_flags = summaries["early_stopping"].astype(str).str.lower().eq("true")
wrong_completed = ((~early_stopping_flags) & (summaries["epochs_completed"] != 20)).sum()
config_columns = [
    "label", "branches", "optimizer", "lr", "momentum", "weight_decay",
    "schedule", "warmup_epochs", "normalization", "batch_size", "dropout",
    "augmentation", "early_stopping", "patience", "epochs_budget", "split_seed",
]
config_variants = summaries.groupby("experiment_id")[config_columns].nunique(dropna=False)
conflicting_configs = config_variants.index[config_variants.gt(1).any(axis=1)].tolist()

print("Số cấu hình đã có:", len(seed_sets), "/", len(expected_ids))
print("Cấu hình còn thiếu:", missing_ids)
print("Cấu hình ngoài protocol:", unexpected_ids)
print("Cấu hình không có đúng seed 42/2026:", wrong_seed_sets)
print("Số dòng thiếu test accuracy:", missing_test_rows)
print("Số dòng test không hữu hạn / sai 10k ảnh:", invalid_test_rows, "/", wrong_test_size)
print("Số dòng sai checkpoint / ngân sách / số epoch:", wrong_checkpoint, "/", wrong_budget, "/", wrong_completed)
print("Cấu hình khác metadata giữa hai seed:", conflicting_configs)

all_checks = [
    missing_ids, unexpected_ids, wrong_seed_sets, missing_test_rows, invalid_test_rows,
    wrong_test_size, wrong_checkpoint, wrong_budget, wrong_completed, conflicting_configs,
]
if RUN_ALL_CONFIGS and any(all_checks):
    raise RuntimeError("Lần chạy ALL chưa tạo đủ 26 cấu hình × 2 seed có test")